<a target="_blank" href="https://colab.research.google.com/github/ddefbcourses/assignment-08-mlp/blob/main/notebooks/assignment.ipynb">
<img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/>
</a>

# Aprendizado de Máquina

Nesta versão da atividade utilizaremos o dataset CIFAR-10.

Características do dataset:

- 60.000 imagens RGB
- 10 classes
- imagens 32×32
- 3 canais de cor

Importante:

O carregamento do dataset pode ser realizado utilizando:

```python
from tensorflow.keras.datasets import cifar10

(X_train, y_train), (X_test, y_test) = cifar10.load_data()
```

Após o carregamento:

```python
print(X_train.shape)
```

Saída esperada:

```python
(50000, 32, 32, 3)
```

Onde:

- 50000 - número de imagens;
- 32 × 32 - dimensão espacial;
- 3 - canais RGB.

Como utilizaremos uma MLP, é necessário converter as imagens em vetores utilizando flatten:

```python
X_train = X_train.reshape(X_train.shape[0], -1)
X_test = X_test.reshape(X_test.shape[0], -1)
```

Após o flatten:

```python
print(X_train.shape)
```

Saída esperada:

```python
(50000, 3072)
```

Isso ocorre porque:

```python
32 × 32 × 3 = 3072
```

# Objetivos

Nesta atividade você irá:

- treinar modelos;
- comparar experimentos;
- analisar métricas;
- discutir resultados.


Nesta atividade utilizaremos MLflow para:

- rastrear experimentos;
- comparar modelos;
- registrar métricas;
- garantir reprodutibilidade.

In [1]:
import warnings

warnings.filterwarnings("ignore")

In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import mlflow

In [3]:
mlflow.set_experiment(
    "assignment"
)

2026/05/22 13:33:51 INFO mlflow.store.db.utils: Creating initial MLflow database tables...
2026/05/22 13:33:52 INFO mlflow.store.db.utils: Updating database tables
2026/05/22 13:33:54 INFO mlflow.tracking.fluent: Experiment with name 'assignment' does not exist. Creating a new experiment.


<Experiment: artifact_location='file:///c:/Users/Matheus/Cesar/Machine_learning/atividade-04-deep-learning-i-Matheuslh/notebooks/mlruns/1', creation_time=1779467634416, experiment_id='1', last_update_time=1779467634416, lifecycle_stage='active', name='assignment', tags={}, trace_location=None, workspace='default'>

# Questão 1

Implemente uma função `load_data(seed)` que:

- carregue o dataset CIFAR-10 utilizando `tensorflow.keras.datasets.cifar10.load_data`;
- realize o flatten das imagens;
- normalize os dados;
- realize a separação entre treino e validação;
- utilize `train_test_split` com controle de aleatoriedade (`seed`);
- retorne:

```python
X_train, X_val, y_train, y_val
```

já normalizados e preparados para treinamento.

Além disso, responda:

1. Qual o formato original das imagens?
2. Quantas features cada imagem possui após o flatten?
3. Por que o flatten é necessário para uma MLP?
4. Qual a importância da normalização para o treinamento?

**Solução**:

In [5]:
from tensorflow.keras.datasets import cifar10
from sklearn.model_selection import train_test_split

def load_data(seed):
    # 1. Carregar o dataset original (focando no conjunto de treino para a divisão)
    (X_train_full, y_train_full), _ = cifar10.load_data()
    
    # 2. Realizar o flatten das imagens (transformar 32x32x3 em 3072)
    X_train_full = X_train_full.reshape(X_train_full.shape[0], -1)
    
    # 3. Normalizar os dados (converter para float e escalar entre 0 e 1)
    X_train_full = X_train_full.astype('float32') / 255.0
    
    # 4. Separação entre treino e validação (utilizando 20% para validação)
    X_train, X_val, y_train, y_val = train_test_split(
        X_train_full, 
        y_train_full, 
        test_size=0.2, 
        random_state=seed,
        stratify=y_train_full 
    )
    
    return X_train, X_val, y_train, y_val

# Resposta

- Questão 1: 32 x 32 pixels com 3 canais rgbs, (32, 32, 3).
- Questão 2: Depois do Flatten ela passa a possuir 3072 features.
- Questão 3: Porque as MLPs possuem uma arquitetura baseada em camadas totalmente conectadas. A camada de entrada dessas redes foi projetada para receber tensores unidimensionais, onde cada neurônio de entrada corresponde a uma única característica (feature) independente. Como as imagens são originalmente estruturas bidimensionais ou tridimensionais, o flatten se faz obrigatório para desdobrar esses pixels em uma única linha contínua, permitindo que cada pixel/canal seja mapeado diretamente para um neurônio da camada de entrada.
- Questão 4: Estabilidade numérica e otimização, algoritmos de descida de gradiente (como SGD ou Adam) convergem muito mais rápido quando as features estão em escalas semelhantes. Prevenção de saturação de funções de ativação, funções de ativação como Sigmoide ou Tanh saturam quando recebem valores muito altos ou muito baixos na entrada, o que interrompe o fluxo de aprendizado dos neurônios. E simetria na superfície de custo, a normalização ajuda a tornar a função de perda mais esférica e menos alongada.

# Questão 2

Implemente a função:

```python
train_mlp(
    X_train,
    y_train,
    activation,
    hidden_layers,
    learning_rate,
    seed
)
```

## Requisitos

Sua implementação deve:

- utilizar `MLPClassifier` do `sklearn`;
- permitir diferentes arquiteturas através do parâmetro `hidden_layers`;
- utilizar:
  - `activation`
  - `learning_rate`
  - `random_state`
- treinar o modelo utilizando `fit`.

A função deve retornar o modelo treinado.

Além disso, responda:

1. Quantos parâmetros existem na primeira camada?
2. Qual a função da ativação ReLU?
3. Por que MLPs possuem muitos parâmetros ao trabalhar com imagens?

**Solução**:

In [ ]:
from sklearn.neural_network import MLPClassifier

def train_mlp(X_train, y_train, activation, hidden_layers, learning_rate, seed):
    
    model = MLPClassifier(
        hidden_layer_sizes=hidden_layers,
        activation=activation,
        learning_rate_init=learning_rate,
        random_state=seed,
        max_iter=100, # Definido um limite padrão de épocas para evitar loops infinitos
        verbose=True  # Permite acompanhar o progresso do treinamento no console
    )
    
    # Treinando o modelo
    # .ravel() transforma a matriz coluna do y_train em um vetor plano exigido pelo sklearn
    model.fit(X_train, y_train.ravel())
    
    return model

# Resposta

- Questão 1: O número de parâmetros na primeira camada depende diretamente do número de neurônios que você escolher para a primeira camada oculta.A fôrmula seria: Total = (3072*H1)+H1
- Questão 2: f(x) = max(0,x)
- Questão 3: Isso acontece devido à natureza totalmente conectada das camadas de uma MLP. Em uma MLP, cada único pixel e cada canal de cor são tratados como uma feature isolada e independente. Cada uma dessas features precisa se conectar individualmente com todos os neurônios da camada seguinte. Como imagens geram vetores de entrada massivos, a multiplicação cruzada com as camadas ocultas faz o número de parâmetros explodir.

# Questão 3

Implemente a função:

```python
evaluate(model, X_test, y_test)
```

Ela deve:

- realizar predições;
- calcular:
  - accuracy;
  - precision;
  - recall;
  - f1-score.

Utilize `sklearn.metrics`.

Além disso:

- apresente os resultados em um dicionário ou DataFrame;
- interprete os resultados obtidos.

Responda:

1. O que a accuracy representa?
2. Qual a diferença entre precision e recall?
3. Em quais situações o f1-score é importante?

**Solução**:

In [ ]:
import pandas as pd
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

def evaluate(model, X_test, y_test):
    # 1. Realizar as predições
    y_pred = model.predict(X_test)
    y_true = y_test.ravel() # Garante que y_true seja um vetor unidimensional
    
    # 2. Calcular as métricas
    # Nota: Como o CIFAR-10 possui 10 classes, precisamos definir o parâmetro 'average'.
    # Usamos o 'macro' para calcular a métrica de cada classe individualmente e tirar a média aritmética simples.
    accuracy = accuracy_score(y_true, y_pred)
    precision = precision_score(y_true, y_pred, average='macro', zero_division=0)
    recall = recall_score(y_true, y_pred, average='macro', zero_division=0)
    f1 = f1_score(y_true, y_pred, average='macro', zero_division=0)
    
    # 3. Apresentar os resultados em um DataFrame do Pandas
    results_dict = {
        'Métrica': ['Acurácia (Accuracy)', 'Precisão (Precision)', 'Revocação (Recall)', 'F1-Score'],
        'Valor Obtido': [accuracy, precision, recall, f1]
    }
    
    df_results = pd.DataFrame(results_dict)
    
    return df_results

# Execução questões 1, 2 e 3

In [ ]:
# 1. Definir as configurações iniciais (Hyperparâmetros)
SEDE_ALEATORIA = 42
ARQUITETURA = (100, 50)      # Uma camada com 100 neurônios, outra com 50
ATIVACAO = 'relu'            # Função de ativação
TAXA_APRENDIZADO = 0.001     # Learning rate

print("🔄 Passo 1: Carregando e preparando os dados...")
# Chamando a primeira função
X_train, X_val, y_train, y_val = load_data(seed=SEDE_ALEATORIA)
print("Dados prontos! Formato do treino:", X_train.shape)

print("\n🏋️‍♂️ Passo 2: Treinando a MLP (isso pode demorar um pouco)...")
# Chamando a segunda função (passando os dados que acabamos de carregar)
modelo_treinado = train_mlp(
    X_train=X_train,
    y_train=y_train,
    activation=ATIVACAO,
    hidden_layers=ARQUITETURA,
    learning_rate=TAXA_APRENDIZADO,
    seed=SEDE_ALEATORIA
)
print("✅ Treinamento concluído!")

print("\n📊 Passo 3: Avaliando o modelo nos dados de validação...")
# Chamando a terceira função (passando o modelo treinado e os dados de VALIDAÇÃO)
tabela_resultados = evaluate(
    model=modelo_treinado, 
    X_test=X_val, 
    y_test=y_val
)

# Exibir a tabela de resultados na tela
print("\n📢 RESULTADOS OBTIDOS:")
display(tabela_resultados) # Se estiver no notebook, 'display' deixa a tabela linda

**Adicione seu texto de solução aqui**.

# Questão 4

Implemente o rastreamento experimental utilizando MLflow.

## Devem ser registrados:

### Parâmetros

- activation
- hidden_layers
- learning_rate
- max_iter
- batch_size

### Métricas

- accuracy
- precision
- recall
- f1_score
- training_time

Utilize:

```python
mlflow.log_param()
mlflow.log_metric()
```

Ao final:

- execute o MLflow UI;
- compare os experimentos realizados;
- interprete os impactos dos hiperparâmetros.

Responda:

1. Qual experimento apresentou melhor desempenho?
2. Qual configuração apresentou maior estabilidade?
3. Qual o benefício do rastreamento experimental?

**Solução**:

In [ ]:
# TODO: implemente

# Questão 5

Compare as funções:

- logistic
- tanh
- relu

## Requisitos

Utilize:

- mesma arquitetura;
- mesmo learning rate;
- mesma seed.

Para cada experimento:

- treine o modelo;
- avalie o modelo;
- registre no MLflow.

Depois compare:

- accuracy;
- convergência;
- estabilidade.

Responda:

1. Qual ativação apresentou melhor convergência?
2. Qual ativação apresentou maior estabilidade?
3. Houve diferenças significativas no treinamento?
4. Por que a ReLU é amplamente utilizada em Deep Learning?

**Solução**:

In [ ]:
# TODO: implemente

# Questão 6

Compare as seguintes arquiteturas:

```python
(32,)
(64,)
(128, 64)
(256, 128)
```

## Requisitos

Para cada arquitetura:

- treine;
- avalie;
- registre no MLflow.

Analise:

- accuracy;
- custo computacional;
- estabilidade;
- overfitting.

Responda:

1. Redes maiores sempre melhoraram os resultados?
2. Qual arquitetura apresentou melhor tradeoff?
3. Houve sinais de overfitting?
4. Qual arquitetura apresentou maior custo computacional?

**Solução**:

In [ ]:
# TODO: implemente

# Questão 7

Compare os seguintes learning rates:

```python
0.1
0.01
0.001
```

## Requisitos

Utilize:

- mesma arquitetura;
- mesma ativação;
- mesma seed.

Para cada experimento:

- treine;
- avalie;
- registre no MLflow.

Analise:

- estabilidade;
- convergência;
- accuracy;
- comportamento da loss.

Responda:

1. Qual learning rate apresentou melhor desempenho?
2. Qual apresentou maior instabilidade?
3. O que acontece quando o learning rate é muito alto?
4. O que acontece quando o learning rate é muito baixo?

In [ ]:
# TODO: implemente

# Questão 8

Com base nos experimentos realizados, escreva uma discussão contendo:

- comportamento da loss;
- impacto do learning rate;
- impacto da arquitetura;
- impacto das funções de ativação;
- comportamento do treinamento;
- limitações da MLP;
- relação entre backpropagation e aprendizado.

Além disso, responda:

1. Qual configuração apresentou melhor resultado final?
2. Quais foram as principais dificuldades observadas?
3. Por que MLPs possuem limitações para imagens?
4. Como o backpropagation contribui para o aprendizado da rede?

In [ ]:
# TODO: implemente